# EnerGIS Test Suite - Interaktiver Test Runner

Dieses Notebook zeigt wie die Tests funktionieren und führt sie interaktiv aus.

## 📋 Was testen wir?

| Test-Kategorie | Datei(en) | Zweck |
|----------------|-----------|-------|
| **Architecture** | `test_v2_architecture.py` | ComponentRegistry, Plugin-System |
| **Workflows** | `test_rolling_workflow.py` | PF, RH, MPC Workflows |
| **Optimization** | `test_investment.py`, `test_capex_persistence.py` | CAPEX, Design-Fixierung |
| **Model Building** | `test_system_builder.py`, `test_bus_balances.py` | Pyomo-Modellbau, Constraints |
| **Data Handling** | `test_exporter.py`, `test_loader_dst.py` | Export, DST-Handling |
| **Physics** | `test_cop_series.py` | COP-Interpolation |
| **Regression** | `test_regression.py` | Bug-Fixes validieren |

**Gesamt:** 13 Test-Dateien, ~2285 Zeilen, 45 Test-Funktionen

---

## 🎯 Wofür brauchen wir Tests?

### 1. **Qualitätssicherung**
- ✅ Code funktioniert wie erwartet
- ✅ Bugs werden früh erkannt
- ✅ Refactoring ist sicher

### 2. **Dokumentation**
- 📚 Tests zeigen wie APIs verwendet werden
- 📚 Beispiele für alle Features

### 3. **Regression Prevention**
- 🛡️ Alte Bugs kommen nicht zurück
- 🛡️ Breaking Changes werden erkannt

### 4. **Continuous Integration**
- 🔄 Automatische Tests bei jedem Commit
- 🔄 Pull Requests werden validiert

## 1. Setup

In [ ]:
# Auto-Setup: Projekt-Root finden
from pathlib import Path
import sys
import os

def find_project_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / '.git').exists() and (candidate / 'energis').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Projekt-Root: {PROJECT_ROOT}")

## 2. Test-Framework prüfen

Die Tests verwenden **pytest** - das Standard-Testing-Framework für Python.

In [ ]:
# Prüfe ob pytest installiert ist
try:
    import pytest
    print(f"✅ pytest installiert: Version {pytest.__version__}")
    HAVE_PYTEST = True
except ImportError:
    print("❌ pytest nicht installiert")
    print("   Installation: pip install pytest")
    HAVE_PYTEST = False

# Prüfe ob Pyomo installiert ist (für Optimierungs-Tests)
try:
    import pyomo.environ as pyo
    print(f"✅ Pyomo installiert")
    HAVE_PYOMO = True
except ImportError:
    print("⚠️  Pyomo nicht installiert (einige Tests werden übersprungen)")
    HAVE_PYOMO = False

## 3. Test-Struktur erkunden

In [ ]:
import glob

test_files = sorted(glob.glob('tests/test_*.py'))

print(f"📂 Gefundene Test-Dateien: {len(test_files)}\n")

for test_file in test_files:
    path = Path(test_file)
    lines = len(path.read_text().splitlines())
    
    # Count test functions
    content = path.read_text()
    test_count = content.count('def test_')
    
    print(f"{path.name:30s}  {lines:4d} Zeilen  {test_count:2d} Tests")

## 4. Einzelnen Test ausführen

### Beispiel: Architecture Test

Dieser Test prüft ob die ComponentRegistry funktioniert.

In [ ]:
# Test kann auch direkt importiert und ausgeführt werden
from tests.test_v2_architecture import test_1_component_registry

try:
    result = test_1_component_registry()
    print("\n" + "="*70)
    print("✅ TEST ERFOLGREICH!")
    print("="*70)
except Exception as e:
    print(f"\n❌ TEST FEHLGESCHLAGEN: {e}")
    import traceback
    traceback.print_exc()

## 5. Alle Tests mit pytest ausführen

### Variante A: Kommandozeile (empfohlen)

```bash
# Alle Tests ausführen
pytest tests/

# Nur einen Test
pytest tests/test_v2_architecture.py

# Mit Ausgabe
pytest tests/ -v

# Mit Logging
pytest tests/ -v --log-cli-level=INFO

# Nur Tests die 'investment' im Namen haben
pytest tests/ -k investment

# Nur Tests die nicht skipif(not HAVE_PYOMO) sind
pytest tests/ -m "not skipif"
```

### Variante B: Aus Notebook (geht auch!)

In [ ]:
if HAVE_PYTEST:
    # pytest kann auch programmatisch aufgerufen werden
    import subprocess
    import sys
    
    print("🧪 Führe Tests aus...\n")
    
    # Beispiel: Nur config und exporter Tests (schnell)
    result = subprocess.run(
        [sys.executable, '-m', 'pytest', 'tests/test_config_merge.py', 'tests/test_exporter.py', '-v'],
        capture_output=True,
        text=True
    )
    
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    
    if result.returncode == 0:
        print("\n✅ Alle Tests bestanden!")
    else:
        print(f"\n❌ {result.returncode} Test(s) fehlgeschlagen")
else:
    print("❌ pytest nicht installiert. Installation: pip install pytest")

## 6. Spezifische Test-Kategorien

### 6.1 Config Merge Test

In [ ]:
# Config Merge Test kann direkt ausgeführt werden
from tests.test_config_merge import test_load_and_merge_from_subdirectory

try:
    # Mock monkeypatch für Notebook
    class MockMonkeypatch:
        def chdir(self, path):
            os.chdir(path)
    
    test_load_and_merge_from_subdirectory(MockMonkeypatch())
    print("✅ Config Merge Test bestanden!")
except Exception as e:
    print(f"❌ Test fehlgeschlagen: {e}")

### 6.2 Workflow Test (Mini-Version)

Zeigt wie Workflow-Tests funktionieren:

In [ ]:
from energis.run import rolling_horizon as rh
from energis.utils.timeseries import TimeSeriesTable
from datetime import datetime, timedelta

print("🧪 Mini Workflow Test\n")

# 1. Config erstellen (minimal)
config = {
    "run": {"dt_h": 1.0, "solver": "glpk"},
    "scenario": {"run_mode": "PF_ONLY"},
    "site": {"input_xlsx": "dummy.xlsx"},
    "system": {
        "heat_pumps": [{"id": "HP1", "max_th_mw": 10.0}],
    }
}

# 2. Mock-Daten erstellen
index = [datetime(2023, 1, 1) + timedelta(hours=i) for i in range(4)]
table = TimeSeriesTable(
    index=index,
    columns=["waermebedarf_MWth"],
    data={"waermebedarf_MWth": [5.0, 6.0, 7.0, 8.0]}
)

print("✓ Config und Daten erstellt")
print(f"  Zeitschritte: {len(table)}")
print(f"  Wärmebedarf: {table.data['waermebedarf_MWth']} MWth")
print()

# 3. Workflow-Struktur testen (ohne echte Optimierung)
print("✓ Workflow-API verfügbar:")
print(f"  - run_workflow: {hasattr(rh, 'run_workflow')}")
print(f"  - export_workflow_results: {hasattr(rh, 'export_workflow_results')}")
print()
print("✅ Mini Workflow Test bestanden!")
print("   (Vollständige Workflow-Tests: tests/test_rolling_workflow.py)")

## 7. Test Coverage

Was wird getestet?

In [ ]:
test_categories = {
    "Architecture & Registry": ["test_v2_architecture.py"],
    "Workflows (PF/RH/MPC)": ["test_rolling_workflow.py", "test_mpc_basic.py"],
    "Optimization & Investment": ["test_investment.py", "test_capex_persistence.py"],
    "Model Building": ["test_system_builder.py", "test_bus_balances.py"],
    "Data I/O": ["test_exporter.py", "test_loader_dst.py"],
    "Physics & Algorithms": ["test_cop_series.py"],
    "Config & Validation": ["test_config_merge.py", "test_stadtbach_validation.py"],
    "Regression": ["test_regression.py"],
}

print("📊 TEST COVERAGE\n")
print("="*70)

for category, files in test_categories.items():
    print(f"\n{category}:")
    for file in files:
        path = Path(f"tests/{file}")
        if path.exists():
            content = path.read_text()
            test_count = content.count('def test_')
            print(f"  ✓ {file:35s} ({test_count:2d} Tests)")
        else:
            print(f"  ⚠ {file} (nicht gefunden)")

print("\n" + "="*70)
print("\n💡 Alle Tests ausführen: pytest tests/")
print("💡 Mit Coverage: pytest tests/ --cov=energis --cov-report=html")

## 8. CI/CD Integration

In Production werden Tests automatisch ausgeführt:

### GitHub Actions Beispiel

```yaml
# .github/workflows/test.yml
name: Tests

on: [push, pull_request]

jobs:
  test:
    runs-on: ubuntu-latest
    
    steps:
    - uses: actions/checkout@v2
    
    - name: Set up Python
      uses: actions/setup-python@v2
      with:
        python-version: '3.11'
    
    - name: Install dependencies
      run: |
        pip install -r requirements.txt
        pip install pytest pytest-cov
    
    - name: Run tests
      run: pytest tests/ -v --cov=energis
```

## 9. Zusammenfassung

### ✅ Tests sind wichtig für:

1. **Qualitätssicherung** - Code funktioniert korrekt
2. **Dokumentation** - Zeigen wie APIs verwendet werden
3. **Regression Prevention** - Bugs kommen nicht zurück
4. **Refactoring Safety** - Code kann sicher umgebaut werden
5. **CI/CD** - Automatische Validierung bei jedem Commit

### 📚 Test-Ausführung:

**Kommandozeile (empfohlen):**
```bash
pytest tests/                    # Alle Tests
pytest tests/ -v                 # Mit Details
pytest tests/test_exporter.py    # Einzelner Test
pytest tests/ -k investment      # Filter nach Name
```

**Notebook (geht auch!):**
- ✅ Tests können importiert und direkt aufgerufen werden
- ✅ pytest kann über subprocess ausgeführt werden
- ✅ Ideal für interaktives Debugging

### 🎯 EnerGIS Test-Stats:

- **13 Test-Dateien**
- **~2285 Zeilen Test-Code**
- **45+ Test-Funktionen**
- **Coverage:** Architecture, Workflows, Optimization, I/O, Physics

---

**Weitere Ressourcen:**
- pytest Docs: https://docs.pytest.org/
- Test-Dateien: `tests/`
- CI/CD: `.github/workflows/` (falls vorhanden)